# QLoRA fine-tuning of the pension answer model (Colab)
**Status: written but NOT run by the author** (no GPU/internet in the build sandbox). Library APIs change often; if a cell errors, check the current docs for `transformers`, `peft`, `trl`.

Before you start: Runtime -> Change runtime type -> **T4 GPU**. Upload `train.jsonl` and `val.jsonl` (made by `export_chat_format.py`) to the Colab files panel or Google Drive.
What this teaches: *behaviour* (cite `[n]`, refuse when evidence is missing, ignore instructions inside evidence) - NOT pension facts.

In [ ]:
!pip install -q transformers peft trl bitsandbytes datasets accelerate

In [ ]:
from google.colab import drive
drive.mount('/content/drive')          # checkpoints are saved here because free sessions time out
OUT_DIR = '/content/drive/MyDrive/pension-slm'
BASE_MODEL = 'PUT_A_SMALL_INSTRUCT_MODEL_NAME_HERE'   # e.g. a 1B-4B instruct model; check its licence first
TRAIN_FILE, VAL_FILE = '/content/train.jsonl', '/content/val.jsonl'

In [ ]:
from datasets import load_dataset
ds = load_dataset('json', data_files={'train': TRAIN_FILE, 'validation': VAL_FILE})
ds = ds.remove_columns([c for c in ds['train'].column_names if c != 'messages'])   # keep only the chat messages
print(ds)
print(ds['train'][0]['messages'][-1]['content'])

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map='auto')

In [ ]:
from peft import LoraConfig
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type='CAUSAL_LM', target_modules='all-linear')

In [ ]:
from trl import SFTConfig, SFTTrainer
cfg = SFTConfig(output_dir=OUT_DIR, num_train_epochs=2, per_device_train_batch_size=2, gradient_accumulation_steps=8,
                learning_rate=2e-4, logging_steps=10, eval_strategy='epoch', save_strategy='epoch', max_length=2048, fp16=True)
trainer = SFTTrainer(model=model, args=cfg, train_dataset=ds['train'], eval_dataset=ds['validation'], peft_config=lora, processing_class=tok)
trainer.train()
trainer.save_model(OUT_DIR + '/adapter')

## After training (on your laptop)
1. Download the `adapter` folder from Drive.
2. Ollama can load a LoRA adapter on top of its matching base model through a `Modelfile` (`FROM <base>` and `ADAPTER <path>`), **or** you merge the adapter into the base model and convert it to GGUF with llama.cpp. Check the current Ollama docs for which formats your version accepts - this was not tested here.
3. `ollama create pension-slm -f Modelfile`, then set `OLLAMA_MODEL=pension-slm` in `.env`.
4. Evaluate: compare base model vs fine-tuned on `data/slm/test.jsonl` and `eval_set.json` before you trust it.